In [1]:
import os
import json
import torch
import torch.nn.functional as F
from llm.gpt import Model

In [2]:
data_dir = "../data/eval"
seq_length = 1024
vocab_size = 50304
embed_dim = 1024
num_heads = 16
query_heads_per_kv = 2
num_blocks = 8
device = "cpu"

if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

loss = torch.nn.CrossEntropyLoss()
model = Model(
    seq_length=seq_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_blocks=num_blocks,
    query_heads_per_kv=query_heads_per_kv,
).to(device)

In [3]:
@torch.inference_mode()
def eval_generate(model, prompts):
    return model(prompts)

In [ ]:
_, _, files = next(os.walk(data_dir))
prompt_beginning = torch.tensor([50256, 24361, 25, 220], dtype=torch.long, device=device)  # "Question: "
prompt_ending = torch.tensor([198, 33706, 25, 220], dtype=torch.long, device=device) #"\nAnswer: "
number_of_correct = 0
total = 0
for file in files:
    with open(f"{data_dir}/{file}", "r") as f:
        data = list(json.load(f))
        for element in data:
            total += 1
            prompts = []
            choice_lengths = []
            question = torch.tensor(element["question"], dtype=torch.long, device=device)

            for choice in element["options"]:
                choice = torch.tensor(choice, dtype=torch.long, device=device)
                prompt = torch.cat((prompt_beginning, question, prompt_ending, choice), dim=0).to(device)
                prompt = F.pad(prompt, pad=(0, int(seq_length - prompt.size(0))), value=50256)
                prompts.append(prompt)
                choice_lengths.append(int(choice.size(0)))

            prompts = torch.stack(prompts, dim=0).to(device)
            output = eval_generate(model, prompts)
            choice_index_start = len(prompt_beginning) +  int(question.size(0)) + len(prompt_ending)

            prompts = prompts[:, choice_index_start:]
            output = output[:, (choice_index_start - 1):]
            output = F.log_softmax(output, dim=-1)
            output = torch.gather(output, dim=-1, index=prompts.unsqueeze(dim=-1)).squeeze(-1)
            sliced_outputs_mask = torch.eq(prompts, 50256)
            output_gather = torch.masked_fill(output, sliced_outputs_mask, 0)
            total_log_probablity = output_gather.sum(dim=-1)
            model_correct_answer = torch.argmax(total_log_probablity, dim=-1)

            if model_correct_answer.item() == element["answer"]:
                number_of_correct += 1

accuracy = number_of_correct / total 
print(f"Model Accuracy: {accuracy*100:.2f}%")

KeyboardInterrupt: 

In [20]:
output_gather.shape

torch.Size([4, 1003])